In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class SimpleEncoder(nn.Module):
    def __init__(self, in_dim=128, feat_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, feat_dim)
        )

    def forward(self, x):
        z = self.net(x)
        print(z.size())
        z = F.normalize(z, dim=1)  # cosine similarity를 위해 정규화
        return z


class MemoryBank:
    def __init__(self, num_samples, feat_dim, device):
        self.device = device
        self.bank = torch.randn(num_samples, feat_dim, device=device)
        self.bank = F.normalize(self.bank, dim=1)

    @torch.no_grad()
    def update(self, indices, features):
        """
        indices: 현재 batch 샘플들의 dataset index
        features: encoder로부터 얻은 feature [B, D]
        """
        self.bank[indices] = features.detach()

    def get_all(self):
        return self.bank

def contrastive_loss_with_memory_bank(features, indices, memory_bank, temperature=0.07):
    """
    features: [B, D] current batch embeddings
    indices: [B] dataset index of current batch
    memory_bank: MemoryBank object
    """
    bank_features = memory_bank.get_all()  # [N, D]

    # 현재 batch와 memory bank 전체 사이의 similarity
    logits = torch.matmul(features, bank_features.T) / temperature  # [B, d] %*% [N, d]^t = [B, N]
    
    # 같은 샘플 index를 positive로 간주
    labels = indices
    loss = F.cross_entropy(logits, labels)
    return loss

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

num_samples = 10000   # num_samples
in_dim = 128
feat_dim = 64
batch_size = 32

encoder = SimpleEncoder(in_dim=in_dim, feat_dim=feat_dim).to(device)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)
memory_bank = MemoryBank(num_samples=num_samples, feat_dim=feat_dim, device=device)

# dummy batch
x = torch.randn(batch_size, in_dim).to(device)
indices = torch.randint(0, num_samples, (batch_size,), device=device)

# forward
features = encoder(x) # [B, d]

# loss
loss = contrastive_loss_with_memory_bank(features, indices, memory_bank)

# backward
optimizer.zero_grad()
loss.backward()
optimizer.step()

# memory bank update
with torch.no_grad():
    updated_features = encoder(x)
    memory_bank.update(indices, updated_features)

print("loss:", loss.item())

torch.Size([32, 64])
torch.Size([32, 64])
loss: 10.334576606750488
